In [ ]:
import faiss

In [ ]:
import torch

In [ ]:
import numpy as np

In [ ]:
# pip install pymupdf
# pip install superkmeans
# pip install ollama

In [ ]:
print("Torch version:", torch.__version__)
print("Built with CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# from sentence_transformers import SentenceTransformer

In [ ]:
import re
from collections import defaultdict, Counter
from pathlib import Path

import json
import random
import re
from collections import defaultdict, Counter
from pathlib import Path

import numpy as np

In [ ]:
from ollama import chat
OLLAMA_MODEL = "llama3.2:3b"

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# METADATA_FILE = "metadata.pkl"

RANDOM_SEED = 42

TARGET_QUESTIONS = 40

# ------------------------------------------------------------
# Question type quotas
# ------------------------------------------------------------

QUESTION_TYPE_QUOTA = {
    "single_chunk": 10,
    "multi_chunk": 10,
#     "multi_page": 10,
    "whole_document": 5,
    "difficult_synthesis": 5,
    "mixed": 10,
}


In [ ]:
# ------------------------------------------------------------
# Question style quotas
#
# These are targets rather than absolute requirements because
# the generator may reject some questions during validation.
# ------------------------------------------------------------

STYLE_QUOTA = {
    "conceptual": 10,
    "detail": 8,
    "comparative": 6,
    "reasoning": 10,
    "summary": 6,
}


# ============================================================
# RANDOM SEED
# ============================================================

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


# Reading document by each page

- Ensure that we are getting the necessary metadata such as the file name, title, author, pages, page no. 
- This can help in searching based on metadata
- Can also aid in expanding the context of the retrieved chunks before passing to the LLM

In [ ]:
# from pathlib import Path
# for pdf_path in Path(r"D:\RAG\pdf files").glob("*.pdf"):
#     print(pdf_path)

In [ ]:
import fitz
from langchain_core.documents import Document
from pathlib import Path

docs = []

for pdf_path in Path(r"D:\RAG\pdf files").glob("*.pdf"):
    print(pdf_path)
    pdf = fitz.open(pdf_path)
    pdf_metadata = pdf.metadata
    print(pdf_metadata)
#     toc = pdf.get_toc()
#     print(toc)

    for page_num, page in enumerate(pdf):
        docs.append(
            Document(
                page_content=page.get_text(),
                metadata={
                    "source": str(pdf_path),
                    "filename": pdf_path.name,
                    "page": page_num + 1,
                    "page_count": len(pdf),
                    "title": pdf_metadata.get("title"),
                    "author": pdf_metadata.get("author"),
                },
            )
        )

In [ ]:
import gc
gc.collect()

# Chunking 

- Add chunk metadata such as the source, page no, chunk index
- This allows easy search of all the chunks that are a part of a specific page of a PDF

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)

In [ ]:
len(chunks)

In [ ]:
# Assign chunk indices within each document
doc_chunk_counter = {}

for chunk in chunks:

    doc = chunk.metadata["source"]

    if doc not in doc_chunk_counter:
        doc_chunk_counter[doc] = 0

    chunk.metadata["chunk_index"] = doc_chunk_counter[doc]
    doc_chunk_counter[doc] += 1

In [ ]:
chunks[3560]

# Creating Metadata for document chunks

In [ ]:
metadata = []

for chunk in chunks:
    metadata.append({
    "text": chunk.page_content,
    "source": chunk.metadata["source"],
    "page": chunk.metadata["page"],
    "chunk_index": chunk.metadata["chunk_index"]
})

In [ ]:
metadata[0]

In [ ]:
from collections import defaultdict
# ============================================================
# BUILD DOCUMENT STRUCTURE
# ============================================================

# source -> list of chunks
documents = defaultdict(list)

for chunk in metadata:
    documents[chunk["source"]].append(chunk)


# Sort chunks within every document
for source in documents:

    documents[source].sort(
        key=lambda x: (
            x["page"],
            x.get("chunk_index", 0)
        )
    )


print(f"Found {len(documents)} documents")


# ============================================================
# BUILD PAGE STRUCTURE
# ============================================================

# (source, page) -> chunks
pages = defaultdict(list)

for chunk in metadata:

    key = (
        chunk["source"],
        chunk["page"]
    )

    pages[key].append(chunk)


for key in pages:

    pages[key].sort(
        key=lambda x: x.get("chunk_index", 0)
    )


# ============================================================
# BUILD DOCUMENT -> PAGES
# ============================================================

document_pages = defaultdict(list)

for source, page in pages.keys():

    document_pages[source].append(page)


for source in document_pages:

    document_pages[source] = sorted(
        document_pages[source]
    )


In [ ]:
# ============================================================
# HELPERS
# ============================================================

def get_text(chunk):

    return chunk["text"].strip()


def page_text(source, page):

    chunks = pages[(source, page)]

    return "\n".join(
        get_text(c)
        for c in chunks
    )


def document_text(source):

    chunks = documents[source]

    return "\n".join(
        get_text(c)
        for c in chunks
    )


def safe_json(text):

    """
    Extract JSON even if the model accidentally wraps
    the response in markdown.
    """

    text = text.strip()

    # Remove ```json ... ```
    text = re.sub(
        r"^```json\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\s*```$",
        "",
        text
    )

    return json.loads(text)


# ============================================================
# LLM CALL
# ============================================================

def ask_llm(prompt):

    response = chat(
        model=OLLAMA_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        format="json"
    )

    return safe_json(
        response["message"]["content"]
    )


In [ ]:
import pickle

with open("metadata.pkl","wb") as f:
    pickle.dump(metadata,f)

In [ ]:
# faiss.write_index(index, "docs.index")

In [ ]:
index = faiss.read_index("docs.index")

# Generating questions for retrieval eval

## Stratified Sampling

In [ ]:
# ============================================================
# STRATIFIED DOCUMENT SAMPLING
# ============================================================

def sample_documents():

    """
    Ensures that the entire corpus has a chance to contribute
    to the evaluation set.

    We allocate at least one candidate to every document,
    then use the remaining candidates proportional to the
    document size.
    """

    sources = list(documents.keys())

    print("\nDocument distribution:")
    print("-" * 60)

    for source in sources:

        print(
            Path(source).name,
            "->",
            len(documents[source]),
            "chunks"
        )

    return sources


# ============================================================
# SAMPLE SINGLE CHUNKS
# ============================================================

def sample_single_chunk_evidence():

    candidates = []

    for source, chunks in documents.items():

        # Prefer chunks that have reasonable text length
        valid = [
            c for c in chunks
            if len(c["text"].strip()) >= 200
        ]

        if not valid:
            continue

        # Pick one representative chunk per document
        chunk = random.choice(valid)

        candidates.append({
            "source": source,
            "type": "single_chunk",
            "chunks": [chunk],
            "pages": [chunk["page"]]
        })

    random.shuffle(candidates)

    return candidates


# ============================================================
# SAMPLE MULTI-CHUNK EVIDENCE
# ============================================================

def sample_multi_chunk_evidence():

    candidates = []

    for source, chunks in documents.items():

        if len(chunks) < 3:
            continue

        # Random starting positions
        possible = range(
            0,
            max(1, len(chunks) - 2)
        )

        starts = list(possible)

        random.shuffle(starts)

        for start in starts[:3]:

            selected = chunks[
                start:start + 3
            ]

            if len(selected) < 2:
                continue

            candidates.append({
                "source": source,
                "type": "multi_chunk",
                "chunks": selected,
                "pages": sorted(
                    set(
                        c["page"]
                        for c in selected
                    )
                )
            })

    random.shuffle(candidates)

    return candidates


# ============================================================
# SAMPLE MULTI-PAGE EVIDENCE
# ============================================================

def sample_multi_page_evidence():

    candidates = []

    for source, page_numbers in document_pages.items():

        if len(page_numbers) < 2:
            continue

        # Two-page windows
        for i in range(
            len(page_numbers) - 1
        ):

            selected_pages = page_numbers[
                i:i + 2
            ]

            selected_chunks = []

            for page in selected_pages:

                selected_chunks.extend(
                    pages[(source, page)]
                )

            if not selected_chunks:
                continue

            candidates.append({
                "source": source,
                "type": "multi_page",
                "chunks": selected_chunks,
                "pages": selected_pages
            })

        # Three-page windows
        for i in range(
            len(page_numbers) - 2
        ):

            selected_pages = page_numbers[
                i:i + 3
            ]

            selected_chunks = []

            for page in selected_pages:

                selected_chunks.extend(
                    pages[(source, page)]
                )

            if not selected_chunks:
                continue

            candidates.append({
                "source": source,
                "type": "multi_page",
                "chunks": selected_chunks,
                "pages": selected_pages
            })

    random.shuffle(candidates)

    return candidates


# ============================================================
# SAMPLE WHOLE DOCUMENTS
# ============================================================

def sample_whole_document_evidence():

    candidates = []

    for source, chunks in documents.items():

        pages_for_doc = document_pages[source]

        if len(chunks) < 5:
            continue

        # We don't necessarily send the entire PDF to the LLM.
        #
        # Instead select representative chunks from the
        # beginning, middle and end.

        positions = [
            0,
            len(chunks) // 2,
            len(chunks) - 1
        ]

        selected = []

        for pos in positions:

            chunk = chunks[pos]

            if chunk not in selected:
                selected.append(chunk)

        candidates.append({
            "source": source,
            "type": "whole_document",
            "chunks": selected,
            "pages": sorted(
                set(
                    c["page"]
                    for c in selected
                )
            )
        })

    random.shuffle(candidates)

    return candidates


# ============================================================
# SAMPLE DIFFICULT SYNTHESIS
# ============================================================

def sample_difficult_evidence():

    candidates = []

    for source, chunks in documents.items():

        if len(chunks) < 10:
            continue

        # Select chunks from different portions of the document.
        #
        # This intentionally creates non-contiguous evidence.

        n = len(chunks)

        positions = [
            int(n * 0.15),
            int(n * 0.50),
            int(n * 0.85)
        ]

        selected = []

        for pos in positions:

            selected.append(
                chunks[pos]
            )

        candidates.append({
            "source": source,
            "type": "difficult_synthesis",
            "chunks": selected,
            "pages": sorted(
                set(
                    c["page"]
                    for c in selected
                )
            )
        })

    random.shuffle(candidates)

    return candidates


## Question styles

In [ ]:
# ============================================================
# GENERATE QUESTION STYLE
# ============================================================

def choose_style():

    available = []

    for style, quota in STYLE_QUOTA.items():

        available.extend(
            [style] * quota
        )

    return random.choice(available)


# ============================================================
# GENERATE QUESTION
# ============================================================

def generate_question(evidence, style):

    source = evidence["source"]
    evidence_type = evidence["type"]

    context_parts = []

    for chunk in evidence["chunks"]:

        context_parts.append(
            f"""
===== PAGE {chunk['page']} =====
===== CHUNK {chunk.get('chunk_index', 'NA')} =====

{chunk['text']}
"""
        )

    context = "\n".join(context_parts)

    if evidence_type == "single_chunk":

        requirement = """
The question must be answerable primarily from ONE
specific chunk.
"""

    elif evidence_type == "multi_chunk":

        requirement = """
The question MUST require information from at least
TWO chunks.

Do not create a question that can be answered using
only one of the chunks.
"""

    elif evidence_type == "multi_page":

        requirement = """
The question MUST require information from at least
TWO DIFFERENT PAGES.

Ideally the answer should connect or synthesize
information across the pages.
"""

    elif evidence_type == "whole_document":

        requirement = """
Create a question that tests understanding of the
document as a whole.

It should require synthesis of the document rather
than a simple fact from one page.
"""

    elif evidence_type == "difficult_synthesis":

        requirement = """
The evidence comes from non-contiguous parts of the
document.

Create a difficult question that requires connecting
information from multiple parts of the document.
"""

    else:

        requirement = """
Create a meaningful RAG question.
"""

    style_instruction = {

        "conceptual": """
Focus on understanding a concept, principle,
definition, mechanism, or relationship.
""",

        "detail": """
Focus on a specific factual detail, condition,
number, property, exception, or technical fact.
""",

        "comparative": """
Compare two or more concepts, methods, approaches,
conditions, or outcomes present in the evidence.
""",

        "reasoning": """
Require reasoning or inference from the evidence.
The answer should not simply copy one sentence.
""",

        "summary": """
Ask for a concise synthesis of the major information
contained in the provided evidence.
"""
    }[style]

    prompt = f"""
You are creating a high-quality benchmark for evaluating
a Retrieval-Augmented Generation system.

SOURCE DOCUMENT:
{Path(source).name}

EVIDENCE:

{context}

QUESTION TYPE:
{evidence_type}

QUESTION STYLE:
{style}

{requirement}

{style_instruction}

GENERAL REQUIREMENTS:

1. The question must be answerable using the supplied evidence.
2. Do not introduce outside knowledge.
3. Do not ask vague questions such as:
   "What is this document about?"
4. The question must be meaningful for testing RAG retrieval.
5. The answer must be objectively supported by the evidence.
6. For multi-page questions, information from multiple pages
   must actually be necessary.
7. For multi-chunk questions, information from multiple chunks
   must actually be necessary.
8. Do not mention page numbers in the question.
9. Do not mention "the provided text" or "the excerpt".
10. Write a natural question that a real user might ask.
11. Provide a concise but complete ground-truth answer.

Return ONLY this JSON:

{{
    "question": "...",
    "ground_truth_answer": "...",
    "difficulty": "easy|medium|hard"
}}
"""

    try:

        result = ask_llm(prompt)

        result["source"] = source
        result["type"] = evidence_type
        result["style"] = style

        result["gold_pages"] = evidence["pages"]

        result["gold_chunks"] = [
            c.get("chunk_index")
            for c in evidence["chunks"]
        ]

        return result

    except Exception as e:

        print(
            "Generation failed:",
            e
        )

        return None




## Validate Questions

In [ ]:
# ============================================================
# VALIDATE QUESTION
# ============================================================

def validate_question(question):

    source = question["source"]

    gold_chunks = question["gold_chunks"]

    chunk_map = {
        c.get("chunk_index"): c
        for c in documents[source]
    }

    context = ""

    for chunk_id in gold_chunks:

        chunk = chunk_map.get(chunk_id)

        if chunk:

            context += f"""
                        ===== PAGE {chunk['page']} =====

                        {chunk['text']}
                        """

    prompt = f"""
            You are validating a synthetic RAG evaluation question.

            QUESTION:
            {question["question"]}

            GROUND TRUTH ANSWER:
            {question["ground_truth_answer"]}

            SOURCE EVIDENCE:
            {context}

            Determine:

            1. Is the question answerable from the evidence?
            2. Is the answer supported by the evidence?
            3. Is the question specific and meaningful?
            4. Does it genuinely test retrieval?
            5. For multi-chunk/multi-page questions, does it require
               multiple pieces of evidence?
            6. Is the question free from unsupported outside knowledge?

            Return ONLY:

            {{
                "valid": true,
                "score": 1,
                "reason": "..."
            }}

            Score:

            5 = excellent benchmark question
            4 = good
            3 = questionable
            2 = poor
            1 = invalid
            """

    try:

        result = ask_llm(prompt)

        return (
            result.get("valid", False)
            and result.get("score", 0) >= 4
        )

    except Exception as e:

        print(
            "Validation failed:",
            e
        )

        return False


In [ ]:
import numpy.random as random

## Creating candidates

In [ ]:
# ============================================================
# CREATE ALL CANDIDATES
# ============================================================

print("\nCreating stratified evidence samples...")

single_candidates = (
    sample_single_chunk_evidence()
)

multi_chunk_candidates = (
    sample_multi_chunk_evidence()
)

# multi_page_candidates = (
#     sample_multi_page_evidence()
# )

whole_document_candidates = (
    sample_whole_document_evidence()
)

difficult_candidates = (
    sample_difficult_evidence()
)


print(
    "Single chunk candidates:",
    len(single_candidates)
)

print(
    "Multi chunk candidates:",
    len(multi_chunk_candidates)
)

# print(
#     "Multi page candidates:",
#     len(multi_page_candidates)
# )

print(
    "Whole document candidates:",
    len(whole_document_candidates)
)

print(
    "Difficult candidates:",
    len(difficult_candidates)
)


# ============================================================
# ALLOCATE EVIDENCE TYPES
# ============================================================

candidate_pools = {

    "single_chunk":
        single_candidates,

    "multi_chunk":
        multi_chunk_candidates,

#     "multi_page":
#         multi_page_candidates,

    "whole_document":
        whole_document_candidates,

    "difficult_synthesis":
        difficult_candidates,

    "mixed":
        (
            single_candidates
            + multi_chunk_candidates
#             + multi_page_candidates
        )
}


## Generating questions

In [ ]:
# ============================================================
# GENERATE QUESTIONS
# ============================================================

evaluation_dataset = []
used_questions = set()

total_attempts = 0
total_accepted = 0
total_rejected = 0

QUESTIONS_TO_GENERATE = 40

for qtype, quota in QUESTION_TYPE_QUOTA.items():

    # --------------------------------------------------------
    # Stop once requested number of questions is reached
    # --------------------------------------------------------

    if len(evaluation_dataset) >= QUESTIONS_TO_GENERATE:
        break

    # If generating only one question, don't need the
    # full quota for each question type.
    remaining = (
        QUESTIONS_TO_GENERATE
        - len(evaluation_dataset)
    )

    target_for_type = min(
        quota,
        remaining
    )

    print("\n" + "=" * 80)
    print(
        f"QUESTION TYPE: {qtype.upper()} "
        f"| TARGET: {target_for_type}"
    )
    print("=" * 80)

    pool = candidate_pools[qtype]

    if not pool:

        print(
            f"⚠ No candidates available for {qtype}"
        )

        continue

    attempts = 0

    while (
        len(evaluation_dataset) < QUESTIONS_TO_GENERATE
        and
        sum(
            1
            for q in evaluation_dataset
            if q["type"] == qtype
        ) < target_for_type
        and
        attempts < target_for_type * 5
    ):

        attempts += 1
        total_attempts += 1

        # ----------------------------------------------------
        # Select evidence
        # ----------------------------------------------------

        evidence = random.choice(pool)

        style = choose_style()

        source_name = Path(
            evidence["source"]
        ).name

        print("\n" + "-" * 80)

        print(
            f"Attempt       : {total_attempts}"
        )

        print(
            f"Type          : {qtype}"
        )

        print(
            f"Style         : {style}"
        )

        print(
            f"Source        : {source_name}"
        )

        print(
            f"Pages         : {evidence['pages']}"
        )

        print(
            f"Chunks        : {len(evidence['chunks'])}"
        )

        print("-" * 80)

        # ----------------------------------------------------
        # Generate
        # ----------------------------------------------------

        question = generate_question(
            evidence,
            style
        )

        if question is None:

            print(
                "❌ Generation failed"
            )

            total_rejected += 1

            continue

        print(
            "\nGenerated question:"
        )

        print(
            f"  {question['question']}"
        )

        print(
            "\nGround-truth answer:"
        )

        print(
            f"  {question['ground_truth_answer']}"
        )

        print(
            f"\nDifficulty: "
            f"{question.get('difficulty', 'unknown')}"
        )

        # ----------------------------------------------------
        # Duplicate check
        # ----------------------------------------------------

        normalized = re.sub(
            r"\W+",
            " ",
            question["question"].lower()
        ).strip()

        if normalized in used_questions:

            print(
                "\n⚠ DUPLICATE → rejected"
            )

            total_rejected += 1

            continue

        # ----------------------------------------------------
        # Validate
        # ----------------------------------------------------

        print(
            "\nValidating question..."
        )

        valid = validate_question(
            question
        )

        if not valid:

            print(
                "❌ VALIDATION FAILED → rejected"
            )

            total_rejected += 1

            continue

        # ----------------------------------------------------
        # Accept
        # ----------------------------------------------------

        used_questions.add(
            normalized
        )

        question["id"] = (
            len(evaluation_dataset) + 1
        )

        evaluation_dataset.append(
            question
        )

        total_accepted += 1

        # ----------------------------------------------------
        # Print useful information
        # ----------------------------------------------------

        print("\n" + "✅" * 20)

        print(
            "QUESTION ACCEPTED"
        )

        print(
            f"ID            : {question['id']}"
        )

        print(
            f"Type          : {question['type']}"
        )

        print(
            f"Style         : {question['style']}"
        )

        print(
            f"Difficulty     : "
            f"{question.get('difficulty', 'unknown')}"
        )

        print(
            f"Source         : {source_name}"
        )

        print(
            f"Gold pages     : "
            f"{question['gold_pages']}"
        )

        print(
            f"Gold chunks    : "
            f"{question['gold_chunks']}"
        )

        print(
            f"Questions      : "
            f"{len(evaluation_dataset)}/"
            f"{QUESTIONS_TO_GENERATE}"
        )

        print(
            f"Accepted       : {total_accepted}"
        )

        print(
            f"Rejected       : {total_rejected}"
        )

        print(
            f"Total attempts : {total_attempts}"
        )

        print("✅" * 20)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 80)
print("GENERATION COMPLETE")
print("=" * 80)

print(
    f"Requested questions : "
    f"{QUESTIONS_TO_GENERATE}"
)

print(
    f"Generated questions : "
    f"{len(evaluation_dataset)}"
)

print(
    f"Total attempts      : "
    f"{total_attempts}"
)

print(
    f"Accepted            : "
    f"{total_accepted}"
)

print(
    f"Rejected             : "
    f"{total_rejected}"
)

print(
    f"Acceptance rate      : "
    f"{(
        total_accepted / total_attempts * 100
        if total_attempts > 0
        else 0
    ):.1f}%"
)

print("\nQuestion types:")

type_counts = Counter(
    q["type"]
    for q in evaluation_dataset
)

for qtype, count in type_counts.items():

    print(
        f"  {qtype:25s}: {count}"
    )

print("\nQuestion styles:")

style_counts = Counter(
    q["style"]
    for q in evaluation_dataset
)

for style, count in style_counts.items():

    print(
        f"  {style:25s}: {count}"
    )

print("\nSources represented:")

source_counts = Counter(
    q["source"]
    for q in evaluation_dataset
)

for source, count in source_counts.items():

    print(
        f"  {Path(source).name:40s}: {count}"
    )

In [ ]:
# ============================================================
# CAP AT TARGET
# ============================================================
TARGET_QUESTIONS = 40
evaluation_dataset = evaluation_dataset[
    :TARGET_QUESTIONS
]


In [ ]:
# ============================================================
# SAVE
# ============================================================

output_file = (
    "rag_evaluation_dataset.json"
)

with open(
    output_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        evaluation_dataset,
        f,
        indent=2,
        ensure_ascii=False
    )



In [ ]:
# ============================================================
# SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("EVALUATION DATASET CREATED")
print("=" * 70)

print(
    "Questions:",
    len(evaluation_dataset)
)

print("\nQuestion types:")

type_counts = Counter(
    q["type"]
    for q in evaluation_dataset
)

for k, v in type_counts.items():

    print(
        f"{k:25s} {v}"
    )


print("\nQuestion styles:")

style_counts = Counter(
    q["style"]
    for q in evaluation_dataset
)

for k, v in style_counts.items():

    print(
        f"{k:25s} {v}"
    )


print("\nDocuments represented:")

doc_counts = Counter(
    q["source"]
    for q in evaluation_dataset
)

for source, count in doc_counts.items():

    print(
        f"{Path(source).name:40s} {count}"
    )


print(
    f"\nSaved to: {output_file}"
)